In [0]:
-- Seeing thr metadata from INFORMATION_SCHEMA
SELECT *
FROM information_schema.columns;

-- distinct table name
SELECT
    DISTINCT TABLE_NAME
FROM information_schema.columns;

-- SUBQUE
-- query insoide another query
-- find the product which has sales higher than average sales
SELECT 
    DISTINCT productname,
    sales
FROM (
    SELECT
        productname,
        sales,
        avg(coalesce(sales, 0)) OVER() AS averageSales
    FROM workspace.default.sales
)t
WHERE sales > averageSales;

-- rank product based on their toal sales
SELECT
    DISTINCT productname,
    sum(sales) OVER(PARTITION BY productname ORDER BY sales DESC) as totalSales
FROM workspace.default.sales
ORDER BY totalSales DESC;

-- another way
SELECT 
    productname,
    sum(sales) as totalSales
FROM workspace.default.sales
GROUP BY productname
ORDER BY totalSales DESC;

-- third approach
SELECT
    *,
    dense_rank() OVER(ORDER BY totalsales DESC) AS rank
FROM (
    SELECT 
        productname,
        sum(sales) as totalSales
    FROM workspace.default.sales
    GROUP BY productname
);

-- using subquery in SELECT
-- only SCALAR subqueries are allowed inside SELECT i.e. those queries which only return one value
SELECT
    cust_id,
    first_name,
    (SELECT count(*) FROM workspace.default.orders) AS totalOrder 
FROM workspace.default.customers;

--- sales > avg(sales)
SELECT 
    DISTINCT productname,
    sales
FROM workspace.default.sales
WHERE sales > (SELECT avg(sales) FROM workspace.default.sales);

-- Details of all the orders who made their order from Germany
SELECT
    *
FROM workspace.default.orders
WHERE cust_id IN (
                SELECT
                    cust_id
                FROM workspace.default.customers
                WHERE country = 'Germany'
);


-- create me an employee table with emp_id, name, gender and salary
-- CREATE TABLE IF NOT EXISTS workspace.default.employee(
--     emp_id INT NOT NULL,
--     name STRING,
--     gender CHAR(1),
--     salary DOUBLE
-- )

-- INSERT INTO workspace.default.employee
-- VALUES (1, 'John', 'M', 50000),
--        (2, 'Jane', 'F', 60000),
--        (3, 'Bob', 'M', 70000),
--        (4, 'Alice', 'F', 80000),
--        (5, 'Charlie', 'M', 90000);

-- check if any female employee has a salary more than male employee
-- this works in many SQL engine but not in databricks
-- SELECT *
-- FROM workspace.default.employee
-- WHERE gender = 'F'
-- AND salary > ANY (SELECT salary FROM workspace.default.employee WHERE gender = 'M');
SELECT *
FROM workspace.default.employee
WHERE gender = 'F'
AND salary >  (SELECT min(salary) FROM workspace.default.employee WHERE gender = 'M');

-- EXISTS
-- order made from germany
SELECT
    *
FROM workspace.default.orders o
WHERE EXISTS (
            SELECT 1
            FROM workspace.default.customers c
            WHERE country = 'Germany'
            AND o.cust_id = c.cust_id
)

